# Week 2 · Pandas Cleaning: raw shipments to a validated "silver" table

# Requirements: pip install pandas numpy
# Uses: the `zoro` package at the repo root, and `data/shipments.csv` from Week 1.

This notebook finds and fixes the **planted** data-quality issues in the raw dataset
(duplicates, `NaN` weights, `NaN` lane distances), then gates the result with a
**validation suite of 12 checks**: the same verification habit that will gate models
and agents later. It ends by printing a pass count and the silver row count.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
p = pathlib.Path.cwd()
while not (p / "zoro").is_dir() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import pandas as pd
import numpy as np
from zoro import data

repo = pathlib.Path.cwd()
while not (repo / "zoro").is_dir() and repo != repo.parent:
    repo = repo.parent

ship_path = repo / "data" / "shipments.csv"
if ship_path.exists():
    raw = pd.read_csv(ship_path, parse_dates=["planned_departure", "planned_arrival", "actual_arrival"])
    lanes = pd.read_csv(repo / "data" / "lanes.csv")
    carriers = pd.read_csv(repo / "data" / "carriers.csv")
    print("Loaded committed CSVs from data/")
else:
    raw = data.shipments(100_000, seed=42)
    lanes = data.lanes(20, seed=11)
    carriers = data.carriers(20, seed=7)
    print("data/ not found - regenerated in-memory from zoro.data (seed 42)")
print("shipments:", raw.shape, "| lanes:", lanes.shape, "| carriers:", carriers.shape)

### What "silver" means

In the medallion architecture you will meet properly in Weeks 21 to 24, **bronze** is the
raw landing data, **silver** is cleaned and conformed, and **gold** is aggregated for
consumption. This week you build silver: a table with a known schema, known
invariants, and a *passing* validation suite. The habit to install is that cleaning is
not a one-off, it is a re-runnable recipe with a gate.

### Step 1, Profile: shape, dtypes, summary stats

Profile before you fix. You cannot decide how to impute a missing value until you know
how many there are and where they live.

In [ ]:
print("shape:", raw.shape)
print()
print("dtypes:")
print(raw.dtypes)
print()
print("numeric summary:")
print(raw[["weight_kg", "value_usd", "delay_hours"]].describe().round(2).to_string())
print()
print("null counts per column:")
print(raw.isna().sum()[raw.isna().sum() > 0])

### Step 2: Find the planted issues

The generator deliberately planted three problems for you: **duplicate rows**
(`pd.concat` of a 0.2% sample), **`NaN` weights** (a 0.3% sample), and **`NaN` lane
distances** (5% of lanes). Count them precisely, a number, not a feeling.

In [ ]:
n_dupes      = int(raw.duplicated().sum())
n_nan_weight = int(raw["weight_kg"].isna().sum())
n_nan_dist   = int(lanes["distance_km"].isna().sum())
print(f"duplicate shipment rows   : {n_dupes:,}")
print(f"NaN weight_kg (shipments) : {n_nan_weight:,}")
print(f"NaN distance_km (lanes)   : {n_nan_dist:,}")

### Step 3, Deeper profiling: find the *un*planted issues

The planted three are the obvious ones. A real data engineer looks further. Check for
impossible values the generator does *not* plant, non-positive weights, a
`planned_arrival` that precedes departure, and the class imbalance (~80% on time / ~20% late)
that will make Week 3's classification accuracy a liar.

In [ ]:
print("non-positive weight_kg:", int((raw["weight_kg"] <= 0).sum()))
print("planned_arrival <= departure:", int((raw["planned_arrival"] <= raw["planned_departure"]).sum()))
print("delay_hours > 100 (extreme late):", int((raw["delay_hours"] > 100).sum()))
print("on-time share:", round(float(raw["is_on_time"].mean()), 4))
print("unique commodities:", raw["commodity"].nunique(), "| unique carriers:", raw["carrier_id"].nunique(), "| unique lanes:", raw["lane_id"].nunique())

### Step 4: The cleaning decisions

Each fix is a *decision with a rationale*, and every decision is recorded:

1. **Drop exact duplicates**: they are the same shipment twice; keeping them inflates
   every downstream count.
2. **Impute `NaN` weight with the median weight *of the same commodity***, weight
   varies hugely by cargo type (electronics vs. machinery), so a per-commodity median
   is more honest than a global one.
3. **Impute `NaN` lane distance with the global median distance**: distance has no
   strong categorical grouping here, so the median is a robust default.
4. **Drop non-positive weights and impossible transit rows** (defensive; none are
   planted) rather than silently carry garbage forward.

In [ ]:
before_rows = len(raw)
ships = raw.drop_duplicates().copy()
n_dupes_removed = before_rows - len(ships)

ships["weight_kg"] = ships["weight_kg"].fillna(
    ships.groupby("commodity")["weight_kg"].transform("median"))

lanes = lanes.copy()
lanes["distance_km"] = lanes["distance_km"].fillna(lanes["distance_km"].median())

n_bad_weight = int((ships["weight_kg"] <= 0).sum())
ships = ships[ships["weight_kg"] > 0].reset_index(drop=True)

n_bad_transit = int((ships["planned_arrival"] <= ships["planned_departure"]).sum())
ships = ships[ships["planned_arrival"] > ships["planned_departure"]].reset_index(drop=True)

print(f"duplicates removed      : {n_dupes_removed:,}")
print(f"weight_kg NaNs imputed  : {n_nan_weight:,}")
print(f"distance_km NaNs imputed: {n_nan_dist:,}")
print(f"non-positive weights    : {n_bad_weight:,}")
print(f"bad transit rows        : {n_bad_transit:,}")
print("cleaned shipments:", ships.shape)

### Step 5: The validation suite (the gate)

A cleaned table is a *claim*; a validation suite is the *proof*. Write the invariants
as executable checks, and require them all to pass before you call the data "silver."
This is the same verification discipline (an eval tied to a requirement) that the
whole program builds on, see `reference/knowledge-base/01-ai-engineering-discipline.md`.

In [ ]:
results = []
def check(name, cond):
    results.append((name, bool(cond)))
    print(("PASS  " if cond else "FAIL  ") + name)

check("no duplicate rows", ships.duplicated().sum() == 0)
check("no NaN weight_kg", ships["weight_kg"].isna().sum() == 0)
check("all weights positive", (ships["weight_kg"] > 0).all())
check("no NaN delay_hours", ships["delay_hours"].isna().sum() == 0)
check("no NaN is_on_time", ships["is_on_time"].isna().sum() == 0)
check("value_usd non-negative", (ships["value_usd"] >= 0).all())
check("carrier_id referential integrity", ships["carrier_id"].isin(carriers["carrier_id"]).all())
check("lane_id referential integrity", ships["lane_id"].isin(lanes["lane_id"]).all())
check("is_on_time is boolean", set(ships["is_on_time"].unique()) <= {True, False})
check("delay_hours within plausible range", ships["delay_hours"].between(-48, 240).all())
check("planned_arrival after departure", (ships["planned_arrival"] > ships["planned_departure"]).all())
check("lanes have no NaN distance", lanes["distance_km"].isna().sum() == 0)

n_pass = sum(1 for _, ok in results if ok)
print()
print("CHECKS_PASSED:", n_pass, "of", len(results))

### Step 6: Persist the silver table and write the report

The silver table, the cleaned lanes, and a human-readable report land in
`data/silver/`. The report is the *audit trail*: what you found, what you did, and why
a stranger should be able to reproduce the table and audit each decision.

In [ ]:
silver_dir = repo / "data" / "silver"
silver_dir.mkdir(parents=True, exist_ok=True)
ships.to_csv(silver_dir / "shipments_silver.csv", index=False)
lanes.to_csv(silver_dir / "lanes_silver.csv", index=False)
carriers.to_csv(silver_dir / "carriers_silver.csv", index=False)

report = [
    "# Silver Pipeline - Cleaning Report", "",
    f"- Input rows: {before_rows:,}",
    f"- Duplicates removed: {n_dupes_removed:,}",
    f"- weight_kg NaNs imputed (median by commodity): {n_nan_weight:,}",
    f"- distance_km NaNs imputed (median): {n_nan_dist:,}",
    f"- Non-positive weights dropped: {n_bad_weight:,}",
    f"- Bad transit rows dropped: {n_bad_transit:,}",
    f"- Output rows: {len(ships):,}",
    "",
    "## Validation", "",
    f"- Checks passed: {n_pass} of {len(results)}",
]
(silver_dir / "validation-report.md").write_text("\n".join(report) + "\n")
print("Saved silver tables + validation report to", str(silver_dir))

### The metric

Two numbers close the notebook: the validation gate and the silver row count.

In [ ]:
print("CHECKS_PASSED:", n_pass, "of", len(results))
print("SILVER_ROW_COUNT:", len(ships))